# 🔬 DermaLens: Dual-Backbone Hybrid Skin Lesion Classifier
### End-to-End Training, Evaluation & Visual Explainability in Google Colab

> [!CAUTION]
> **UNDER ACTIVE DEVELOPMENT — RESEARCH PROTOTYPE**
> DermaLens is an investigational screening-support prototype. It does not constitute a medical diagnosis.

This notebook packages the complete DermaLens training pipeline into a single, self-contained environment:
1. **Dual Visual Backbones:** EfficientNet-B4 (17.5M) + Swin Transformer V2 (86.9M)
2. **Clinical Patient Metadata MLP:** 13-dimensional one-hot encoded metadata (Age, Sex, Anatomical Site)
3. **Loss & Regularization:** Multi-class Focal Loss, Label Smoothing, CutMix augmentation, and Auxiliary Metadata Loss
4. **Optimization:** AdamW with Swin layer-wise LR decay, Cosine Annealing, and Stochastic Weight Averaging (SWA)
5. **Evaluation & Explainability:** Full classification report, confusion matrix, and Grad-CAM++ attention heatmaps

---
**Recommended Runtime:** `Runtime` → `Change runtime type` → **T4 GPU** (or A100/V100).


In [ ]:
# Step 1: Install required dependencies
!pip install -q timm albumentations opencv-python-headless scikit-learn pandas matplotlib seaborn kaggle

import os
import sys
import time
import random
import json
import shutil
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import AveragedModel, SWALR

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit

# Check CUDA device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Compute Device : {device}")
if torch.cuda.is_available():
    print(f"✓ GPU Model      : {torch.cuda.get_device_name(0)}")
    print(f"✓ Device Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Warning: Running on CPU. For fast training, enable GPU under Runtime -> Change runtime type.")


In [ ]:
# Step 2: Global Configuration & Hyperparameters
class Config:
    # Diagnostic Classes (ISIC 2019 8-class standard)
    CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
    NUM_CLASSES = len(CLASS_NAMES)
    CLASS_FULL = {
        "MEL": "Melanoma",
        "NV": "Melanocytic Nevus",
        "BCC": "Basal Cell Carcinoma",
        "AK": "Actinic Keratosis",
        "BKL": "Benign Keratosis",
        "DF": "Dermatofibroma",
        "VASC": "Vascular Lesion",
        "SCC": "Squamous Cell Carcinoma",
    }
    ALL_SITES = [
        "anterior torso", "head/neck", "lateral torso", "lower extremity",
        "oral/genital", "palms/soles", "posterior torso", "upper extremity", "unknown"
    ]
    
    # Image & Batch Settings
    IMG_SIZE = 384
    BATCH_SIZE = 8             # Optimized for Colab 15GB GPU (set to 16 if using A100)
    NUM_WORKERS = 2
    GRAD_ACCUM_STEPS = 2      # Effective batch size = 16
    
    # Training Duration & Learning Rates
    EPOCHS = 10                # Set to 25-50 for full convergence on ISIC 2019
    LR = 1e-4
    WEIGHT_DECAY = 0.01
    SWIN_LR_DECAY = 0.65       # Layer-wise decay for Swin Transformer
    COSINE_T_MAX = 10
    SWA_START = 6              # Stochastic Weight Averaging start epoch
    
    # Loss & Regularization
    FOCAL_GAMMA = 2.0
    FOCAL_ALPHA = [1.0] * 8
    LABEL_SMOOTHING = 0.1
    CUTMIX_PROB = 0.30
    META_AUX_WEIGHT = 0.3      # Weight of auxiliary loss on metadata branch
    
    # Preprocessing
    AGE_NORM_FACTOR = 90.0     # Standardized min-max normalization factor
    USE_AMP = torch.cuda.is_available()
    SEED = 42
    OUTPUT_DIR = Path("./dermalens_output")

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=Config.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()
print("✓ Hyperparameters and seed initialized.")


### Step 3: Dataset Ingestion — Option B (Kaggle ISIC 2019) vs Quick-Start Demo
Choose your dataset mode below:
* **Option B: Kaggle ISIC 2019 (`USE_KAGGLE = True`)**:
  1. Download your `kaggle.json` API token from [Kaggle Account Settings](https://www.kaggle.com/settings).
  2. Run the cell below — it will prompt you to upload `kaggle.json`, download the official ISIC 2019 dataset, extract all 25,331 images, and build stratified `train.csv`, `val.csv`, and `test.csv` manifests automatically!
* **Option C: Quick-Start Demo (`USE_KAGGLE = False`)**:
  - Automatically synthesizes a 184-image mini dermoscopy dataset so you can test the code immediately without downloading gigabytes of data.


In [ ]:
# Set USE_KAGGLE = True to train on the real ISIC 2019 dataset from Kaggle!
USE_KAGGLE = True

train_csv = None
val_csv = None
test_csv = None

if USE_KAGGLE:
    from google.colab import files
    
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / "kaggle.json"
    
    # 1. Setup kaggle.json
    if not kaggle_json.exists():
        print("Please upload your 'kaggle.json' file downloaded from Kaggle (Profile -> Settings -> Create New Token):")
        uploaded = files.upload()
        if "kaggle.json" in uploaded:
            shutil.move("kaggle.json", str(kaggle_json))
            kaggle_json.chmod(0o600)
            print("✓ kaggle.json configured.")
        else:
            print("kaggle.json was not uploaded; falling back to demo generator.")
            USE_KAGGLE = False
    else:
        print("✓ Found existing kaggle.json.")

if USE_KAGGLE:
    isic_dir = Path("./isic2019")
    isic_dir.mkdir(parents=True, exist_ok=True)
    
    # 2. Download ISIC 2019 dataset
    print()
    print("[KAGGLE] Downloading ISIC 2019 challenge dataset...")
    # Attempt competition download first; if not accepted, use the public community dataset
    ret = os.system("kaggle competitions download -c isic-2019 -p ./isic2019 --unzip")
    if ret != 0:
        print("[KAGGLE] Attempting alternative Kaggle dataset mirror...")
        os.system("kaggle datasets download -d andrewmvd/isic-2019 -p ./isic2019 --unzip")
        
    # 3. Locate GroundTruth and Metadata CSVs
    gt_candidates = list(isic_dir.rglob("*GroundTruth*.csv")) + list(isic_dir.rglob("*ground_truth*.csv"))
    meta_candidates = list(isic_dir.rglob("*Metadata*.csv")) + list(isic_dir.rglob("*metadata*.csv"))
    
    if not gt_candidates:
        print("Warning: Could not locate GroundTruth CSV in downloaded files; falling back to demo generator.")
        USE_KAGGLE = False
    else:
        gt_path = gt_candidates[0]
        meta_path = meta_candidates[0] if meta_candidates else None
        print(f"✓ GroundTruth found: {gt_path}")
        if meta_path:
            print(f"✓ Metadata found: {meta_path}")
            
        gt_df = pd.read_csv(gt_path)
        meta_df = pd.read_csv(meta_path) if meta_path else pd.DataFrame()
        
        # Determine training image directory
        img_dirs = [d for d in isic_dir.rglob("*") if d.is_dir() and "train" in d.name.lower()]
        img_root = img_dirs[0] if img_dirs else isic_dir
        print(f"✓ Image directory: {img_root}")
        
        # Build image lookup index
        print("[INDEX] Scanning image files...")
        image_index = {}
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            for f in img_root.rglob(ext):
                image_index[f.stem] = str(f.resolve())
        print(f"✓ Indexed {len(image_index)} images on disk.")
        
        # Merge GroundTruth + Metadata
        if not meta_df.empty and "image" in meta_df.columns:
            merged = pd.merge(gt_df, meta_df, on="image", how="left")
        else:
            merged = gt_df.copy()
            merged["age_approx"] = 45.0
            merged["sex"] = "unknown"
            merged["anatom_site_general"] = "unknown"
            
        # Drop UNK rows if present
        if "UNK" in merged.columns:
            merged = merged[merged["UNK"] != 1.0].copy()
            
        # Determine single class_encoded label (0-7)
        label_cols = [c for c in Config.CLASS_NAMES if c in merged.columns]
        merged["class_encoded"] = merged[label_cols].values.argmax(axis=1)
        merged["target"] = merged["class_encoded"].map(lambda x: Config.CLASS_NAMES[x])
        
        # Map image filepaths
        merged["image_path"] = merged["image"].map(image_index)
        valid_df = merged.dropna(subset=["image_path"]).copy()
        
        # Format metadata column names
        site_col = "anatom_site_general" if "anatom_site_general" in valid_df.columns else "site"
        if site_col in valid_df.columns:
            valid_df["site"] = valid_df[site_col].fillna("unknown").astype(str).str.lower()
        else:
            valid_df["site"] = "unknown"
            
        valid_df["sex"] = valid_df["sex"].fillna("unknown").astype(str).str.lower() if "sex" in valid_df.columns else "unknown"
        valid_df["age_approx"] = pd.to_numeric(valid_df["age_approx"], errors="coerce").fillna(45.0)
        
        print(f"✓ Matched {len(valid_df)} valid images with complete labels.")
        
        # Stratified 80/10/10 Train / Val / Test split
        splitter_tv = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=Config.SEED)
        train_idx, holdout_idx = next(splitter_tv.split(valid_df, valid_df["class_encoded"]))
        
        train_df = valid_df.iloc[train_idx].copy()
        holdout_df = valid_df.iloc[holdout_idx].copy()
        
        splitter_vt = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=Config.SEED)
        val_sub_idx, test_sub_idx = next(splitter_vt.split(holdout_df, holdout_df["class_encoded"]))
        
        val_df = holdout_df.iloc[val_sub_idx].copy()
        test_df = holdout_df.iloc[test_sub_idx].copy()
        
        manifest_dir = isic_dir / "manifests"
        manifest_dir.mkdir(parents=True, exist_ok=True)
        train_csv = manifest_dir / "train.csv"
        val_csv = manifest_dir / "val.csv"
        test_csv = manifest_dir / "test.csv"
        
        train_df.to_csv(train_csv, index=False)
        val_df.to_csv(val_csv, index=False)
        test_df.to_csv(test_csv, index=False)
        
        print(f"[READY] ISIC 2019 Dataset Manifests Created:")
        print(f"  Train : {len(train_df)} images ({train_csv})")
        print(f"  Val   : {len(val_df)} images ({val_csv})")
        print(f"  Test  : {len(test_df)} images ({test_csv})")

# Fallback Option C: Automated Quick-Start Demo Generator
if not USE_KAGGLE:
    print()
    print("[DEMO] Building quick-start synthetic dataset...")
    DATA_ROOT = Path("./dataset_demo")
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    
    def generate_synthetic_lesion(class_name: str, size=(Config.IMG_SIZE, Config.IMG_SIZE)) -> np.ndarray:
        h, w = size
        base_color = np.array([random.randint(180, 230), random.randint(140, 185), random.randint(120, 160)], dtype=np.uint8)
        img = np.full((h, w, 3), base_color, dtype=np.uint8)
        noise = np.random.normal(0, 10, (h, w, 3)).astype(np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        
        center = (w // 2 + random.randint(-20, 20), h // 2 + random.randint(-20, 20))
        axes = (random.randint(60, 110), random.randint(50, 95))
        angle = random.randint(0, 180)
        
        if class_name in ("MEL", "NV"):
            lesion_color = (random.randint(30, 70), random.randint(20, 50), random.randint(10, 40))
        elif class_name in ("BCC", "SCC"):
            lesion_color = (random.randint(120, 160), random.randint(60, 90), random.randint(70, 100))
        elif class_name == "VASC":
            lesion_color = (random.randint(140, 190), random.randint(20, 50), random.randint(40, 70))
        else:
            lesion_color = (random.randint(80, 120), random.randint(60, 90), random.randint(40, 60))
            
        cv2.ellipse(img, center, axes, angle, 0, 360, lesion_color, -1)
        img = cv2.GaussianBlur(img, (25, 25), 0)
        return img
        
    records = []
    for split, count in [("train", 120), ("val", 32), ("test", 32)]:
        split_dir = DATA_ROOT / split
        split_dir.mkdir(parents=True, exist_ok=True)
        for i in range(count):
            cls_idx = i % Config.NUM_CLASSES
            cls_name = Config.CLASS_NAMES[cls_idx]
            img_arr = generate_synthetic_lesion(cls_name)
            img_name = f"ISIC_{split}_{i:04d}_{cls_name}.jpg"
            img_path = split_dir / img_name
            cv2.imwrite(str(img_path), cv2.cvtColor(img_arr, cv2.COLOR_RGB2BGR))
            
            records.append({
                "split": split,
                "image_path": str(img_path.resolve()),
                "class_encoded": cls_idx,
                "target": cls_name,
                "age_approx": random.choice([25, 40, 55, 65, 75, np.nan]),
                "sex": random.choice(["male", "female", "unknown"]),
                "site": random.choice(Config.ALL_SITES),
            })
            
    df_all = pd.DataFrame(records)
    train_csv = DATA_ROOT / "train.csv"
    val_csv   = DATA_ROOT / "val.csv"
    test_csv  = DATA_ROOT / "test.csv"
    
    df_all[df_all["split"] == "train"].drop(columns=["split"]).to_csv(train_csv, index=False)
    df_all[df_all["split"] == "val"].drop(columns=["split"]).to_csv(val_csv, index=False)
    df_all[df_all["split"] == "test"].drop(columns=["split"]).to_csv(test_csv, index=False)
    print(f"✓ Synthetic test dataset created at: {DATA_ROOT}")


In [ ]:
# Step 4: Data Augmentation & Dataloaders
TRAIN_TRANSFORMS = A.Compose([
    A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=45, p=0.6),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

VAL_TRANSFORMS = A.Compose([
    A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

def encode_metadata(age=None, sex=None, site=None) -> torch.Tensor:
    """Encodes tabular metadata into a normalized 13-dimensional tensor."""
    meta = np.zeros(13, dtype=np.float32)
    meta[0] = min(max(float(age), 0.0), Config.AGE_NORM_FACTOR) / Config.AGE_NORM_FACTOR if age is not None and not np.isnan(age) else 0.5
    sex_str = str(sex or "unknown").lower()
    if sex_str == "female":
        meta[1] = 1.0
    elif sex_str == "male":
        meta[2] = 1.0
    else:
        meta[3] = 1.0
        
    site_str = str(site or "unknown").lower()
    matched = False
    for i, s in enumerate(Config.ALL_SITES):
        if s == site_str:
            meta[4 + i] = 1.0
            matched = True
            break
    if not matched:
        meta[4 + Config.ALL_SITES.index("unknown")] = 1.0
    return torch.tensor(meta, dtype=torch.float32)

class DermaLensDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["image_path"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image = self.transform(image=image)["image"]
            
        meta = encode_metadata(row.get("age_approx"), row.get("sex"), row.get("site"))
        label = torch.tensor(int(row["class_encoded"])).long()
        return image, meta, label

# Dataloaders with WeightedRandomSampler for class balance
train_dataset = DermaLensDataset(train_csv, transform=TRAIN_TRANSFORMS)
val_dataset   = DermaLensDataset(val_csv, transform=VAL_TRANSFORMS)
test_dataset  = DermaLensDataset(test_csv, transform=VAL_TRANSFORMS)

labels = train_dataset.df["class_encoded"].values
class_counts = np.bincount(labels, minlength=Config.NUM_CLASSES)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=Config.NUM_WORKERS)
val_loader   = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
test_loader  = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

print(f"✓ Dataloaders ready: {len(train_loader)} batches train | {len(val_loader)} batches val")


In [ ]:
# Step 5: DermaLens Hybrid Model Architecture
class DermaLensHybrid(nn.Module):
    def __init__(self, num_classes=Config.NUM_CLASSES, meta_dim=13, mode='full', pretrained=True):
        super().__init__()
        self.mode = mode
        
        # 1. EfficientNet-B4 Convolutional Backbone (1792-d)
        if mode in ('effnet_only', 'image_only', 'full'):
            self.effnet = timm.create_model('efficientnet_b4', pretrained=pretrained, num_classes=0)
            
        # 2. Swin Transformer V2 Backbone (1024-d)
        if mode in ('swin_only', 'image_only', 'full'):
            self.swin = timm.create_model('swinv2_base_window12to24_192to384.ms_in22k_ft_in1k', pretrained=pretrained, num_classes=0)
            
        # 3. Clinical Metadata MLP (13 -> 64 -> 32-d)
        if mode == 'full':
            self.meta_mlp = nn.Sequential(
                nn.Linear(meta_dim, 64),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, 32),
                nn.ReLU()
            )
            self.meta_aux_head = nn.Linear(32, num_classes) # Auxiliary loss supervisor
            
        fusion_dim = {
            'effnet_only': 1792,
            'swin_only': 1024,
            'image_only': 1792 + 1024,
            'full': 1792 + 1024 + 32
        }[mode]
        
        # 4. Late-Fusion Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
        
        if mode in ('image_only', 'full'):
            self.freeze_backbones()
            
    def freeze_backbones(self):
        """Freeze early backbone layers to avoid catastrophic forgetting during transfer learning."""
        if hasattr(self, 'effnet'):
            for name, param in self.effnet.named_parameters():
                if not name.startswith('blocks.6') and not name.startswith('conv_head') and not name.startswith('bn2'):
                    param.requires_grad = False
        if hasattr(self, 'swin'):
            for name, param in self.swin.named_parameters():
                if not name.startswith('layers.3') and not name.startswith('norm'):
                    param.requires_grad = False
                    
    def forward(self, img, meta=None, return_aux=False):
        features = []
        aux_logits = None
        
        if self.mode in ('effnet_only', 'image_only', 'full'):
            features.append(self.effnet(img))
        if self.mode in ('swin_only', 'image_only', 'full'):
            features.append(self.swin(img))
        if self.mode == 'full':
            if meta is None:
                meta = torch.zeros(img.size(0), 13, device=img.device)
            meta_feat = self.meta_mlp(meta)
            features.append(meta_feat)
            if hasattr(self, 'meta_aux_head'):
                aux_logits = self.meta_aux_head(meta_feat)
                
        logits = self.classifier(torch.cat(features, dim=1))
        if return_aux:
            return logits, aux_logits
        return logits

# Instantiate Model
model = DermaLensHybrid(num_classes=Config.NUM_CLASSES, mode='full', pretrained=True).to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model initialized | Trainable parameters: {trainable_params/1e6:.2f}M / {total_params/1e6:.2f}M total")


In [ ]:
# Step 6: Loss Functions, CutMix & Optimizer
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.alpha = torch.tensor(alpha or [1.0]*8, dtype=torch.float32).to(device)
        self.smoothing = label_smoothing

    def forward(self, logits, targets):
        num_classes = logits.size(-1)
        with torch.no_grad():
            smooth_targets = torch.full_like(logits, self.smoothing / (num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        log_p = F.log_softmax(logits, dim=1)
        p = torch.exp(log_p)
        focal_weight = (1.0 - p) ** self.gamma
        if self.alpha is not None:
            focal_weight = focal_weight * self.alpha.unsqueeze(0)
        return -(focal_weight * smooth_targets * log_p).sum(dim=1).mean()

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
    return x, y[index], lam

def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

# Layer-wise Optimizer Setup
params = []
if hasattr(model, 'effnet'):
    params.append({'params': model.effnet.parameters(), 'lr': Config.LR * 0.1})
if hasattr(model, 'swin'):
    params.append({'params': model.swin.parameters(), 'lr': Config.LR * 0.1 * Config.SWIN_LR_DECAY})
if hasattr(model, 'meta_mlp'):
    params.append({'params': model.meta_mlp.parameters(), 'lr': Config.LR * 3.0})
if hasattr(model, 'meta_aux_head'):
    params.append({'params': model.meta_aux_head.parameters(), 'lr': Config.LR * 3.0})
params.append({'params': model.classifier.parameters(), 'lr': Config.LR})

optimizer = torch.optim.AdamW(params, weight_decay=Config.WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=Config.COSINE_T_MAX, eta_min=1e-6)
criterion = FocalLoss(gamma=Config.FOCAL_GAMMA, alpha=Config.FOCAL_ALPHA, label_smoothing=Config.LABEL_SMOOTHING)

# Stochastic Weight Averaging (SWA)
swa_model = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=Config.LR * 0.5)

print("✓ Optimizer, Schedulers, and Loss Criterion configured.")


In [ ]:
# Step 7: Complete Training Loop
def validate(net, loader):
    net.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, metas, targets in loader:
            imgs, metas = imgs.to(device), metas.to(device)
            outputs = net(imgs, metas)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(targets.numpy())
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return macro_f1, acc

print("Starting DermaLens Hybrid Model Training...")
scaler = torch.amp.GradScaler('cuda', enabled=Config.USE_AMP)
best_f1 = 0.0
best_model_path = Config.OUTPUT_DIR / "best_dermalens_model.pt"

for epoch in range(1, Config.EPOCHS + 1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    
    for step, (images, metas, targets) in enumerate(train_loader):
        images, metas, targets = images.to(device), metas.to(device), targets.to(device)
        
        # Apply CutMix
        use_cutmix = random.random() < Config.CUTMIX_PROB
        if use_cutmix:
            images, targets_cut, lam = cutmix_data(images, targets)
        else:
            targets_cut, lam = targets, 1.0
            
        with torch.amp.autocast('cuda', enabled=Config.USE_AMP):
            output = model(images, metas, return_aux=True)
            logits, meta_aux = output if isinstance(output, tuple) else (output, None)
            
            # Loss calculation
            if use_cutmix:
                main_loss = lam * criterion(logits, targets) + (1 - lam) * criterion(logits, targets_cut)
            else:
                main_loss = criterion(logits, targets)
                
            loss = main_loss
            if meta_aux is not None:
                aux_loss = criterion(meta_aux, targets)
                loss = loss + Config.META_AUX_WEIGHT * aux_loss
                
            loss = loss / Config.GRAD_ACCUM_STEPS
            
        scaler.scale(loss).backward()
        total_loss += loss.item() * Config.GRAD_ACCUM_STEPS
        
        if (step + 1) % Config.GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
    # SWA update
    if epoch >= Config.SWA_START:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        scheduler.step()
        
    val_f1, val_acc = validate(model, val_loader)
    print(f"Epoch [{epoch:02d}/{Config.EPOCHS:02d}] - Loss: {total_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc*100:.2f}%")
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"  [CHECKPOINT] Saved new best model checkpoint (Macro-F1: {best_f1:.4f})")

print()
print(f"[COMPLETE] Training finished! Best validation Macro-F1: {best_f1:.4f}")


In [ ]:
# Step 8: Temperature Scaling Calibration (Softening Overconfident Logits)
# Deep neural networks often make overconfident predictions (e.g. 98% confidence on an incorrect call).
# Temperature Scaling learns a temperature factor T > 0 on validation logits: P_cal = softmax(logits / T).
# This softens overconfident predictions while strictly preserving argmax accuracy and ranking!

class TemperatureScaler:
    def __init__(self, n_classes=Config.NUM_CLASSES):
        self.n_classes = n_classes
        self.global_temp = 1.0
        self.per_class_temps = np.ones(n_classes, dtype=np.float32)

    def fit(self, val_logits: torch.Tensor, val_labels: torch.Tensor):
        val_logits = val_logits.cpu()
        val_labels = val_labels.cpu()
        nll_fn = nn.CrossEntropyLoss()

        # 1. Global Temperature Optimization via grid search
        best_t, best_nll = 1.0, nll_fn(val_logits, val_labels).item()
        for t in np.arange(0.2, 5.0, 0.05):
            loss = nll_fn(val_logits / t, val_labels).item()
            if loss < best_nll:
                best_nll = loss
                best_t = t
        self.global_temp = float(np.round(best_t, 3))

        # 2. Per-Class Temperature Optimization via L-BFGS
        temps = torch.ones(self.n_classes, dtype=torch.float32, requires_grad=True)
        optimizer = torch.optim.LBFGS([temps], lr=0.05, max_iter=200)

        def closure():
            optimizer.zero_grad()
            scaled = val_logits / temps.clamp(min=0.1)
            loss = nll_fn(scaled, val_labels)
            loss.backward()
            return loss

        optimizer.step(closure)
        self.per_class_temps = temps.detach().clamp(min=0.1).numpy()
        return self

    def scale_global(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.global_temp

    def scale_per_class(self, logits: torch.Tensor) -> torch.Tensor:
        temps = torch.from_numpy(self.per_class_temps).to(logits.device).clamp(min=0.1)
        return logits / temps


def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    """Expected Calibration Error (ECE): weighted average gap between confidence and accuracy."""
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(np.float32)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(labels)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (confidences >= lo) & (confidences < hi) if i < n_bins - 1 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (mask.sum() / n) * abs(bin_acc - bin_conf)
    return float(ece)

# 1. Collect validation set logits
print("Collecting validation set logits for temperature calibration...")
model.eval()
val_logits_list, val_labels_list = [], []
with torch.no_grad():
    for imgs, metas, targets in val_loader:
        imgs, metas = imgs.to(device), metas.to(device)
        logits = model(imgs, metas)
        val_logits_list.append(logits.cpu())
        val_labels_list.append(targets)

val_logits_tensor = torch.cat(val_logits_list, dim=0)
val_labels_tensor = torch.cat(val_labels_list, dim=0)

# 2. Fit Temperature Scalers
scaler = TemperatureScaler(n_classes=Config.NUM_CLASSES)
scaler.fit(val_logits_tensor, val_labels_tensor)

# 3. Calculate Calibration Gains
raw_probs = F.softmax(val_logits_tensor, dim=1).numpy()
global_cal_probs = F.softmax(scaler.scale_global(val_logits_tensor), dim=1).numpy()
pc_cal_probs = F.softmax(scaler.scale_per_class(val_logits_tensor), dim=1).numpy()
y_val_np = val_labels_tensor.numpy()

ece_raw = compute_ece(raw_probs, y_val_np)
ece_global = compute_ece(global_cal_probs, y_val_np)
ece_pc = compute_ece(pc_cal_probs, y_val_np)

print("=" * 60)
print("       TEMPERATURE SCALING CALIBRATION RESULTS")
print("=" * 60)
print(f"Optimal Global Temperature (T) : {scaler.global_temp:.3f}")
print("Per-Class Temperatures (T_k)    :")
for name, t_k in zip(Config.CLASS_NAMES, scaler.per_class_temps):
    interp = "Softened (was overconfident)" if t_k > 1.05 else ("Sharpened" if t_k < 0.95 else "Well-calibrated")
    print(f"  {name:<6}: T = {t_k:.3f} ({interp})")

print("-" * 60)
print(f"Expected Calibration Error (ECE):")
print(f"  Raw Logits (Uncalibrated)    : {ece_raw * 100:.2f}%")
print(f"  Global Temperature Scaled    : {ece_global * 100:.2f}%")
print(f"  Per-Class Temperature Scaled : {ece_pc * 100:.2f}%")
error_reduction = ((ece_raw - ece_pc) / max(ece_raw, 1e-5)) * 100
print(f"  Relative Calibration Gain    : {error_reduction:.1f}% error reduction")
print("=" * 60)

# 4. Save calibration parameters to output directory for production deployment
np.save(Config.OUTPUT_DIR / "optimal_temperature.npy", np.array(scaler.global_temp, dtype=np.float32))
np.save(Config.OUTPUT_DIR / "per_class_temperatures.npy", scaler.per_class_temps)
print(f"✓ Saved calibration files to {Config.OUTPUT_DIR}/")

# 5. Plot Reliability Diagram (Calibration Curve)
fig, ax = plt.subplots(figsize=(7, 6))
bin_edges = np.linspace(0, 1, 11)

for p_arr, p_lbl, p_col in [
    (raw_probs, f"Uncalibrated (ECE={ece_raw*100:.1f}%)", "tab:red"),
    (pc_cal_probs, f"Per-Class Calibrated (ECE={ece_pc*100:.1f}%)", "tab:green")
]:
    confs = p_arr.max(axis=1)
    accs = (p_arr.argmax(axis=1) == y_val_np).astype(np.float32)
    bin_confs, bin_accs = [], []
    for i in range(10):
        mask = (confs >= bin_edges[i]) & (confs < bin_edges[i+1])
        if mask.sum() > 0:
            bin_confs.append(confs[mask].mean())
            bin_accs.append(accs[mask].mean())
    ax.plot(bin_confs, bin_accs, marker='o', label=p_lbl, color=p_col, linewidth=2)

ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
ax.set_title("Reliability Diagram (Confidence vs. Real Accuracy)")
ax.set_xlabel("Predicted Confidence")
ax.set_ylabel("Empirical Accuracy")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Step 9: Comprehensive Test Set Evaluation (with Calibrated Probabilities)
# Load best weights
model.load_state_dict(torch.load(best_model_path))
model.eval()

y_preds, y_trues, y_probs = [], [], []
with torch.no_grad():
    for imgs, metas, targets in test_loader:
        imgs, metas = imgs.to(device), metas.to(device)
        logits = model(imgs, metas)
        # Apply per-class temperature scaling to soften overconfident logits
        scaled_logits = scaler.scale_per_class(logits)
        probs = F.softmax(scaled_logits, dim=1).cpu().numpy()
        preds = scaled_logits.argmax(dim=1).cpu().numpy()
        y_preds.extend(preds)
        y_trues.extend(targets.numpy())
        y_probs.extend(probs)

print("=" * 60)
print("     DERMALENS CLASSIFICATION REPORT (CALIBRATED)")
print("=" * 60)
print(classification_report(y_trues, y_preds, target_names=Config.CLASS_NAMES, zero_division=0))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_trues, y_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=Config.CLASS_NAMES, yticklabels=Config.CLASS_NAMES)
plt.title("DermaLens Confusion Matrix (Test Set)")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.tight_layout()
plt.show()


In [ ]:
# Step 10: Visual Explainability (Grad-CAM++ with Calibrated Confidence)
class GradCAMPlusPlus:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)
        
    def save_activation(self, module, input, output):
        self.activations = output
        
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]
        
    def generate(self, img_tensor, meta_tensor, target_class=None):
        self.model.eval()
        output = self.model(img_tensor, meta_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
            
        self.model.zero_grad()
        score = output[0, target_class]
        score.backward(retain_graph=True)
        
        grads = self.gradients[0]
        acts = self.activations[0]
        
        # Grad-CAM++ weighting
        g2 = grads ** 2
        g3 = grads ** 3
        spatial_sum = acts.sum(dim=(1, 2), keepdim=True)
        alpha = g2 / (2 * g2 + spatial_sum * g3 + 1e-7)
        weights = (alpha * F.relu(grads)).sum(dim=(1, 2), keepdim=True)
        cam = (weights * acts).sum(dim=0).cpu().detach().numpy()
        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (Config.IMG_SIZE, Config.IMG_SIZE))
        
        # Raw vs Calibrated Confidence
        raw_prob = F.softmax(output, dim=1)[0, target_class].item()
        cal_output = scaler.scale_per_class(output)
        cal_prob = F.softmax(cal_output, dim=1)[0, target_class].item()
        
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-7), target_class, raw_prob, cal_prob

# Run Grad-CAM++ on sample test image
sample_img, sample_meta, sample_label = test_dataset[0]
img_tensor = sample_img.unsqueeze(0).to(device)
meta_tensor = sample_meta.unsqueeze(0).to(device)

gradcam = GradCAMPlusPlus(model, model.effnet.conv_head)
cam_map, pred_class, raw_conf, cal_conf = gradcam.generate(img_tensor, meta_tensor)

# Visualization
raw_display = sample_img.permute(1, 2, 0).cpu().numpy()
raw_display = (raw_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
heatmap = cv2.applyColorMap(np.uint8(255 * cam_map), cv2.COLORMAP_JET)
heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
overlay = (0.6 * raw_display + 0.4 * heatmap).clip(0, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw_display)
axes[0].set_title(f"Original Input (Ground Truth: {Config.CLASS_NAMES[sample_label]})")
axes[0].axis("off")

axes[1].imshow(cam_map, cmap="jet")
axes[1].set_title("Grad-CAM++ Attention Heatmap")
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title(f"Overlay | Pred: {Config.CLASS_NAMES[pred_class]} | Raw: {raw_conf*100:.1f}% -> Cal: {cal_conf*100:.1f}%")
axes[2].axis("off")

plt.tight_layout()
plt.show()
print(f"✓ Visual explanation generated: Predicted {Config.CLASS_FULL[Config.CLASS_NAMES[pred_class]]}")
print(f"  Confidence: Raw {raw_conf*100:.1f}% -> Calibrated {cal_conf*100:.1f}% (Softened)")
